# Classificação de Expressões Faciais com YOLOv8

Este notebook demonstra todas as etapas para criar, treinar, validar e testar um classificador de expressões faciais usando YOLOv8.

**⚡ Versão Otimizada para Colab Gratuito:**
- Dataset: ~20MB (333 imagens)
- Tempo de treino: ~20 minutos
- GPU: T4 (gratuita)

**Agenda:**
1. Setup e instalações
2. Configuração do ambiente
3. Download do dataset
4. Exploração e análise dos dados
5. Distribuição dos dados
6. Preparação dos dados
7. Treinamento do modelo
8. Avaliação e validação
9. Matriz de confusão
10. Relatório detalhado
11. Testes em novas imagens
12. Análise de resultados
13. Uso em produção
14. Conclusões

## 1. Setup e Instalações

Vamos instalar todas as dependências necessárias.

In [ ]:
# Instalar YOLOv8
!pip install ultralytics -q

# Instalar bibliotecas auxiliares
!pip install opencv-python matplotlib pillow scikit-learn -q

print("✓ Instalações concluídas!")

## 2. Configuração do Ambiente

Vamos configurar o ambiente e verificar se temos GPU disponível.

In [ ]:
import os
import shutil
import subprocess
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import json
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Verificar GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU disponível:")
for gpu in gpu_info:
    print(f"  {gpu}")

# Importar YOLO
from ultralytics import YOLO

print("\n✓ Ambiente configurado com sucesso!")

## 3. Download do Dataset

Vamos clonar o repositório com o dataset do GitHub.

**IMPORTANTE:** Substitua `SEU_USUARIO` e `SEU_REPOSITORIO` pelos seus dados do GitHub.

In [ ]:
# Configurar variáveis do repositório
GITHUB_USER = "SEU_USUARIO"  # Substitua com seu usuário GitHub
GITHUB_REPO = "SEU_REPOSITORIO"  # Substitua com o nome do seu repositório
GITHUB_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
DATASET_PATH = "/content/dataset-sample"

print(f"🔽 Clonando dataset de: {GITHUB_URL}")
print(f"   Tamanho esperado: ~20MB")
print(f"   Tempo esperado: 1-3 minutos\n")

# Remover pastas se já existem
if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)
if os.path.exists("/tmp/repo"):
    shutil.rmtree("/tmp/repo")

# Clonar repositório (apenas o subset)
!git clone --depth 1 {GITHUB_URL} /tmp/repo
!cp -r /tmp/repo/dataset-sample {DATASET_PATH}

print("\n✓ Dataset baixado!")

## 4. Exploração e Análise dos Dados

Vamos entender a estrutura e conteúdo do dataset.

In [ ]:
# Explorar estrutura do dataset
dataset_base = Path(DATASET_PATH)

# Encontrar pastas de train e val
train_dir = dataset_base / "train"
val_dir = dataset_base / "val"

print("Estrutura do dataset:")
print(f"Pasta train existe: {train_dir.exists()}")
print(f"Pasta val existe: {val_dir.exists()}")

# Listar classes
if train_dir.exists():
    classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    print(f"\n✓ Classes encontradas: {classes}")
    print(f"  Total de classes: {len(classes)}")
    
    # Contar imagens por classe
    print("\n📊 Distribuição de imagens:")
    print("\nTreinamento:")
    train_counts = {}
    for class_dir in train_dir.iterdir():
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")))
            train_counts[class_dir.name] = count
            print(f"  {class_dir.name}: {count} imagens")
    
    print("\nValidação:")
    val_counts = {}
    for class_dir in val_dir.iterdir():
        if class_dir.is_dir():
            count = len(list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")))
            val_counts[class_dir.name] = count
            print(f"  {class_dir.name}: {count} imagens")
    
    print(f"\n📈 Totais:")
    print(f"  Treino: {sum(train_counts.values())} imagens")
    print(f"  Validação: {sum(val_counts.values())} imagens")

## 5. Visualização de Amostras

Vamos visualizar algumas imagens de cada classe para entender os dados.

In [ ]:
# Visualizar amostras de cada classe
fig, axes = plt.subplots(len(classes), 3, figsize=(12, 4*len(classes)))
fig.suptitle('Amostras do Dataset - Expressões Faciais', fontsize=16, fontweight='bold')

for idx, class_name in enumerate(classes):
    class_path = train_dir / class_name
    images = sorted(list(class_path.glob("*.jpg")) + list(class_path.glob("*.png")))[:3]
    
    for img_idx, img_path in enumerate(images):
        ax = axes[idx, img_idx] if len(classes) > 1 else axes[img_idx]
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(class_name if img_idx == 0 else "")
        ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Amostras exibidas!")

## 6. Distribuição dos Dados

Visualizamos a distribuição de imagens por classe para entender o balanceamento do dataset.

In [ ]:
# Criar gráfico de distribuição
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Treino
ax = axes[0]
classes_sorted = sorted(train_counts.keys())
counts_sorted = [train_counts[c] for c in classes_sorted]
bars = ax.bar(classes_sorted, counts_sorted, color='steelblue', edgecolor='black')
ax.set_xlabel('Expressão Facial', fontsize=12)
ax.set_ylabel('Número de Imagens', fontsize=12)
ax.set_title('Distribuição - Treino', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10)

# Validação
ax = axes[1]
classes_sorted_val = sorted(val_counts.keys())
counts_sorted_val = [val_counts[c] for c in classes_sorted_val]
bars = ax.bar(classes_sorted_val, counts_sorted_val, color='coral', edgecolor='black')
ax.set_xlabel('Expressão Facial', fontsize=12)
ax.set_ylabel('Número de Imagens', fontsize=12)
ax.set_title('Distribuição - Validação', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Adicionar valores nas barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("✓ Distribuição visualizada!")

## 7. Preparação dos Dados para YOLOv8

YOLOv8 para classificação espera uma estrutura específica. Vamos preparar um arquivo YAML com as configurações.

In [ ]:
# Criar arquivo de configuração (data.yaml)
yaml_content = f"""path: {DATASET_PATH}
train: train
val: val

nc: {len(classes)}
names: {classes}
"""

yaml_path = Path(DATASET_PATH) / "data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✓ Arquivo de configuração criado: {yaml_path}")
print("\nConteúdo:")
print(yaml_content)

## 8. Treinamento do Modelo

Agora vamos treinar o modelo YOLOv8 para classificação de expressões faciais.

**Parâmetros de treinamento:**
- **epochs**: 50 (número de passagens pelo dataset)
- **imgsz**: 224 (tamanho das imagens)
- **batch**: 16 (tamanho do lote, otimizado para Colab)
- **device**: GPU (0)
- **patience**: 10 (parar se não houver melhora)

In [ ]:
# Carregar modelo base (classificação)
model = YOLO('yolov8s-cls.pt')

print("="*60)
print("🚀 INICIANDO TREINAMENTO")
print("="*60)
print(f"Dataset: {DATASET_PATH}")
print(f"Modelo: YOLOv8 Small (Classification)")
print(f"Classes: {len(classes)}")
print(f"Tempo esperado: ~20 minutos")
print("="*60 + "\n")

# Treinar o modelo
results = model.train(
    data=str(DATASET_PATH),
    epochs=50,
    imgsz=224,
    batch=16,
    device=0,
    patience=10,
    save=True,
    verbose=True,
    project='/content/runs',
    name='facial_expressions'
)

print("\n" + "="*60)
print("✓ TREINAMENTO CONCLUÍDO!")
print("="*60)

## 9. Avaliação do Modelo

Vamos avaliar o desempenho do modelo no conjunto de validação.

In [ ]:
# Validar o modelo
print("\n📊 Validando modelo...\n")
metrics = model.val()

print("\n" + "="*60)
print("MÉTRICAS DE VALIDAÇÃO")
print("="*60)
print(f"Top-1 Accuracy: {metrics.top1:.2%}")
print(f"Top-5 Accuracy: {metrics.top5:.2%}")
print("="*60)

print("\n✓ Validação concluída!")

## 10. Matriz de Confusão

Vamos criar uma matriz de confusão para ver como o modelo classifica cada expressão facial.

In [ ]:
# Carregar modelo treinado
model = YOLO('/content/runs/facial_expressions/weights/best.pt')

# Coletar todas as predições de validação
true_labels = []
pred_labels = []
all_val_images = []

for class_dir in val_dir.iterdir():
    if class_dir.is_dir():
        images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in images:
            all_val_images.append((str(img_path), class_dir.name))

print(f"Processando {len(all_val_images)} imagens de validação...\n")

# Fazer predições
predictions = model.predict([img for img, _ in all_val_images], conf=0.5)

for (img_path, true_label), pred in zip(all_val_images, predictions):
    pred_class = pred.names[int(pred.probs.top1)]
    true_labels.append(true_label)
    pred_labels.append(pred_class)

# Criar matriz de confusão
cm = confusion_matrix(true_labels, pred_labels, labels=classes)

# Plotar
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes,
            cbar_kws={'label': 'Contagem'})
plt.xlabel('Predição', fontsize=12)
plt.ylabel('Valor Real', fontsize=12)
plt.title('Matriz de Confusão - Validação', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("✓ Matriz de confusão gerada!")

## 11. Relatório Detalhado por Classe

Vamos visualizar precisão, recall e F1-score para cada classe.

In [ ]:
# Gerar relatório de classificação
report = classification_report(true_labels, pred_labels, 
                               target_names=classes,
                               output_dict=True)

print("\n" + "="*80)
print("RELATÓRIO DETALHADO POR CLASSE")
print("="*80)
print(f"{'Classe':<20} {'Precisão':>12} {'Recall':>12} {'F1-Score':>12} {'Suporte':>10}")
print("-"*80)

for class_name in classes:
    metrics = report[class_name]
    print(f"{class_name:<20} {metrics['precision']:>12.2%} {metrics['recall']:>12.2%} {metrics['f1-score']:>12.2%} {int(metrics['support']):>10}")

print("-"*80)
print(f"{'Acurácia Geral':<20} {report['accuracy']:>12.2%}")
print("="*80)

# Gráfico de barras com métricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Precisão
precision_scores = [report[c]['precision'] for c in classes]
axes[0].bar(classes, precision_scores, color='skyblue', edgecolor='black')
axes[0].set_ylabel('Precisão', fontsize=12)
axes[0].set_title('Precisão por Classe', fontsize=14, fontweight='bold')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Recall
recall_scores = [report[c]['recall'] for c in classes]
axes[1].bar(classes, recall_scores, color='lightcoral', edgecolor='black')
axes[1].set_ylabel('Recall', fontsize=12)
axes[1].set_title('Recall por Classe', fontsize=14, fontweight='bold')
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

# F1-Score
f1_scores = [report[c]['f1-score'] for c in classes]
axes[2].bar(classes, f1_scores, color='lightgreen', edgecolor='black')
axes[2].set_ylabel('F1-Score', fontsize=12)
axes[2].set_title('F1-Score por Classe', fontsize=14, fontweight='bold')
axes[2].set_ylim([0, 1])
axes[2].grid(axis='y', alpha=0.3)
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\n✓ Relatório detalhado gerado!")

## 12. Testes em Novas Imagens

Vamos testar o modelo em imagens do conjunto de validação e exibir os resultados.

In [ ]:
# Pegar algumas imagens de teste (uma por classe)
test_images = []
test_labels = []

for class_dir in val_dir.iterdir():
    if class_dir.is_dir():
        images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        if images:
            test_images.append(str(images[0]))
            test_labels.append(class_dir.name)

# Fazer predições
print(f"🧪 Testando em {len(test_images)} imagens...\n")
predictions = model.predict(test_images, conf=0.5)

# Exibir resultados
cols = min(5, len(test_images))
rows = (len(test_images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
if len(test_images) == 1:
    axes = [axes]
elif rows == 1:
    axes = axes.flatten()
else:
    axes = axes.flatten()

for idx, (pred, label, img_path) in enumerate(zip(predictions, test_labels, test_images)):
    if idx >= len(axes):
        break
    ax = axes[idx]
    img = Image.open(img_path)
    
    # Predição
    pred_class = pred.names[int(pred.probs.top1)]
    pred_conf = pred.probs.top1conf.item()
    
    # Cor baseada em acerto
    correct = "✓" if pred_class == label else "✗"
    color = "green" if pred_class == label else "red"
    
    ax.imshow(img)
    ax.set_title(f"{correct} Real: {label}\nPred: {pred_class} ({pred_conf:.1%})", color=color)
    ax.axis('off')

# Remover axes vazios
for idx in range(len(test_images), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

print("✓ Testes concluídos!")

## 13. Análise de Treinamento

Vamos visualizar as métricas de treinamento ao longo dos epochs.

In [ ]:
# Carregar histórico de treinamento
results_dir = Path("/content/runs/facial_expressions")

# Ler arquivo de resultados
results_csv = results_dir / "results.csv"
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    
    # Plotar histórico de treinamento
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Acurácia
    if 'metrics/accuracy_top1' in df.columns:
        axes[0].plot(df['metrics/accuracy_top1'], label='Top-1 Acc', marker='o', linewidth=2)
        axes[0].set_xlabel('Epoch', fontsize=12)
        axes[0].set_ylabel('Acurácia', fontsize=12)
        axes[0].set_title('Acurácia por Epoch', fontsize=14, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend(fontsize=11)
    
    # Loss
    if 'train/loss' in df.columns:
        axes[1].plot(df['train/loss'], label='Train Loss', marker='o', linewidth=2, color='red')
        if 'val/loss' in df.columns:
            axes[1].plot(df['val/loss'], label='Val Loss', marker='s', linewidth=2, color='orange')
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('Loss', fontsize=12)
        axes[1].set_title('Loss por Epoch', fontsize=14, fontweight='bold')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Gráficos de treinamento exibidos!")
else:
    print("⚠️ Arquivo de resultados não encontrado")

## 14. Uso do Modelo em Produção

Aqui mostramos como usar o modelo treinado para fazer predições em novas imagens.

In [ ]:
def predict_expression(image_path, model, conf_threshold=0.5):
    """
    Faz predição de expressão facial em uma imagem.

    Args:
        image_path: caminho da imagem
        model: modelo YOLOv8
        conf_threshold: limite de confiança

    Returns:
        dict com resultado da predição
    """
    result = model.predict(image_path, conf=conf_threshold)[0]
    
    predicted_class = result.names[int(result.probs.top1)]
    confidence = float(result.probs.top1conf)
    
    # Todas as probabilidades
    all_probs = {result.names[i]: float(result.probs.data[i]) for i in range(len(result.names))}
    
    return {
        'class': predicted_class,
        'confidence': confidence,
        'all_probabilities': all_probs
    }

# Exemplo de uso
print("\n💡 Exemplo de predição:\n")
result = predict_expression(test_images[0], model)
print(f"Classe predita: {result['class']}")
print(f"Confiança: {result['confidence']:.2%}")
print("\nTodas as probabilidades:")
for class_name, prob in sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {class_name:15s}: {prob:6.2%}")

## 15. Conclusões e Próximos Passos

### ✅ Resumo do Projeto:
- ✓ Dataset explorado (333 imagens, 9 classes)
- ✓ Distribuição de dados analisada
- ✓ Modelo YOLOv8 treinado
- ✓ Validação e matriz de confusão
- ✓ Relatório detalhado por classe
- ✓ Testes em imagens reais
- ✓ Pronto para uso em produção

### 🚀 Melhorias Possíveis:
1. **Aumentar dataset**: Adicionar mais imagens para melhor generalização
2. **Data augmentation**: Aplicar transformações (rotação, distorção, etc)
3. **Ajuste de hiperparâmetros**: Testar diferentes taxas de aprendizagem
4. **Ensemble**: Combinar múltiplos modelos
5. **Deploy**: Converter para ONNX ou TFLite para produção

### 📚 Recursos:
- [YOLOv8 Documentação](https://docs.ultralytics.com/)
- [Ultralytics GitHub](https://github.com/ultralytics/ultralytics)
- [Scikit-learn Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)